In [ ]:
import numpy as np
import pandas as pd
import os #Für Arbeit mit Dateipfaden
from sklearn.decomposition import PCA


In [ ]:
import torch #Für Arbeit mit neuronalen Netzen
from esm.models.esmc import ESMC
from esm.utils.constants.models import ESMC_600M
from esm.tokenization import EsmSequenceTokenizer
# EsmSequenceTokenizer ist eine Objektklasse, die Methoden wie .encode() und .decode() hat,
# um zwischen Sequenz und Aminosäure-Tokens zu unterscheiden
from esm.utils import encoding
# encoding ist ein Modul (.py-Datei), das Funktionen wie tokenize_sequence(seq=str, tokenizer=EsmSequenceTokenizer, ...) enthält

esm = ESMC.from_pretrained("esmc_600m").to('cuda')
# Lädt das trainierte (d.h. z.B. optimierte weights und biases durch Minimuerung der Loss-Function) ESMC-Modell mit dem Befehl, das Modell auf der GPU auszuführen
tokenizer = EsmSequenceTokenizer()
# Lädt den Tokenizer als Objekt einer Klasse mit Methoden wie .encode() und .decode(), um zwischen AS-Identität und zugehöriger Token-ID zu übersetzen

In [ ]:
regions = ["CDR_H1", "CDR_H2", "CDR_H3", "CDR_L1", "CDR_L2", "CDR_L3"]
df = (
    pd.read_csv("data/ab_ag_annotated.tsv", sep="\t", usecols=["pdb"] + regions)
    .dropna(subset = regions)
    .drop_duplicates(subset = regions, keep='first')
    .reset_index(drop=True)
)
# Lädt einen dataframe mit pdb-IDs und zugehörigen CDR-Sequenzen. Keine doppelten oder NA Sequenzeinträge

for region in regions:
    to_process = [
       (n, s)
       for n, s in zip(df.pdb, df[region])
       if not os.path.exists(f'…/Target_{n}.npy') # Was macht die Zeile? Was bedeutet der path?
    ]
    with torch.no_grad():
    # no_grad heißt: keine Backpropagation für Anpassung der weights und bias um loss-function zu minimieren (= Modell trainieren), sondern neuronales Netz nur "vorwärts" laufen lassen
    for layer_index in [0:35]
            embedding_space = pd.DataFrame(columns=residues, index=sequences.index)
        for name, seq in to_process:
            tokenized = encoding.tokenize_sequence(seq, tokenizer, add_special_tokens=True).to('cuda')
            # encoding: ein Modul aus dem Paket esm.utils
            # tokenize_sequence: Funktion aus dem Modul encoding: Wandelt AS-Sequenz (str) in das Input-Layer des neuralen Netzes um, indem es jeder AS-Identität (str) eine Token-ID (num) zuweist
            # add_special_tokens=True: fügt am Anfang und Ende der Token-Sequenz je ein Sondertoken ein, um Anfang und Ende für das Modell sichtbar zu machen
            pred = esm.forward(tokenized.unsqueeze(0), repr_layers=[layer_index])
            # tokenized.unsqueeze(0) fügt dem Input-Vektor tokenized eine zusätzliche Dimension hinzu, die der batch_size(= Anzahl der gleichzeitig untersuchten Proteine) entspricht.
            # tokenized wird formal zu einer Matrix, obwohl die zusätzliche Dimension leer ist, weil bei uns: batch_size = 1
            # --> tokenized.unsequeeze(0).shape = [Sequenzlänge, batch_size = 1]
            # notwendig, weil esm.forward() eine solche Matrix erwartet
            # esm.forward(): "schickt" die Input-Matrix durch das Transformer-Netzwerk (d.h. führt Matrix-Multiplikation mit den beim Training eingestellten weights und biases durch)
            # --> pred ist eine Matrix aus den logits des Output-Layers (Reihen entsprechen einzelnen AS in der Sequenzen, mehrere Spalten pro AS repräsentieren eine AS im Ouuput-Layer des Transformers)
            embeddings_hidden = pred.hidden_representations[layer_index].to('cpu').squeeze()
            embeddings_output = pred.embeddings.to('cpu').squeeze()
            # pred.embeddings: prozessiert die Matrix (WIE?) und speichert sie als embeddings ab
            # .to('cpu'): embeddings wird wieder in der cpu gespeichert, da numpy hier arbeitet
            # .squeeze: .unsqueeze() wird wieder rückgängig gemacht
            embed_hid_as_arr = embeddings_hidden.float().detach().numpy()
            embed_out_as_arr = embeddings_output.float().detach().numpy()
            # .float():Klassen der Einträge der Logits werden in floats umgewandelt. GPU hat vorher mit bfloat16 gearbeitet
            # .detach(): Informationen über das zugrungdeliegende Transformer-Netz wird verworfen, da wir nicht an Backpropagation interessiert sind
            # .numpy(): wandelt embeddings von der Klasse tensor aus PyTorch in ein array aus NumPy um
            embed_hid_as_arr = embed_hid_as_arr[1:-1,:]
            embed_out_as_arr = embed_out_as_arr[1:-1,:]
            # Entfernt die SpecialTokens, die in der ersten Code-Zeile zugefügt wurden
            # --> entfernt erste und letzte Reihe des 2D-arrays, alle Spalten bleiben erhalten
            np.save(f"esmc_ansatz/embeddings/{name}_{region}.npy", embed_as_arr)
            # Speichert das NumPy-Array ab
            embed_hid_as_vec = PCA(n_components=1).fit_transform(embed_hid_as_arr.T)
            embed_out_as_vec = PCA(n_components=1).fit_transform(embed_out_as_arr.T)

            # PCA, um Embedding-Dimensionen zu erhalten, aber AS-Anzahl zu quetschen --> Embedding-Matrix in Embedding-Vektor quetschen
            # Kann man auch anders herum PCA nutzen, oder über Means quetschen, oder den cls-Token (special_token) als Vektor für jede Sequenz nehmen?
            # Danach müssen alle embedding-Vektoren in einem dataframe gespeichert werden (1 Zeile pro Sequenz)
            # Danach kmeans auf diesen Raum anwenden

1548
1548
1548
1548
1548
1548
